In [ ]:
import pandas as pd

In [ ]:
train_data = pd.read_csv("../data/raw/vihsd/train.csv")
dev_data = pd.read_csv("../data/raw/vihsd/dev.csv")
test_data = pd.read_csv("../data/raw/vihsd/test.csv")

In [ ]:
# Gom 3 tập vào một dict để lặp
splits = {"train": train_data, "dev": dev_data, "test": test_data}

# Đánh số nhãn của dataset
LABEL_NAMES = {0: "CLEAN", 1: "OFFENSIVE", 2: "HATE"}

In [ ]:
total = train_data.shape[0] + test_data.shape[0] + dev_data.shape[0]
print(total)

In [ ]:
train_data.head()

In [ ]:
train_data.info()
dev_data.info()
test_data.info()

In [ ]:
print(train_data.isna().sum())
print(dev_data.isna().sum())
print(test_data.isna().sum())

In [ ]:
print(train_data["label_id"].value_counts())
print(train_data["label_id"].value_counts(normalize=True))

In [ ]:
print(dev_data["label_id"].value_counts())
print(dev_data["label_id"].value_counts(normalize=True))

In [ ]:
print(test_data["label_id"].value_counts())
print(test_data["label_id"].value_counts(normalize=True))

In [ ]:
# Dòng trùng lặp trong từng tập (bỏ NaN trước, vì duplicated() coi các NaN là trùng nhau)
for name, df in splits.items():
    text = df["free_text"].dropna()
    n_dup = text.duplicated().sum()
    n_repeated_texts = text[text.duplicated(keep=False)].nunique()
    print(
        f"{name}: {n_dup} dòng trùng ({n_dup / len(df):.1%}), "
        f"{n_repeated_texts} câu khác nhau bị lặp lại"
    )

In [ ]:
# Cùng một câu nhưng bị gán nhiều nhãn khác nhau trong train -> bằng chứng nhãn có nhiễu
labels_per_text = train_data.groupby("free_text")["label_id"].nunique()
conflicting = labels_per_text[labels_per_text > 1]
print(f"Số câu trong train bị gán từ 2 nhãn khác nhau trở lên: {len(conflicting)}")

# Bảng: mỗi hàng là một câu mâu thuẫn, mỗi cột là số lần câu đó mang nhãn 0/1/2
# (chỉ xem 15 câu đầu, chỉ xem trên máy, không dán nội dung ra ngoài)
conflict_rows = train_data[train_data["free_text"].isin(conflicting.index)]
print(conflict_rows.groupby("free_text")["label_id"].value_counts().unstack(fill_value=0).head(15))

In [ ]:
# Rò rỉ giữa các tập: câu trong dev/test cũng có mặt trong train
train_texts = set(train_data["free_text"].dropna())

for name in ["dev", "test"]:
    df = splits[name]
    in_train = df["free_text"].isin(train_texts)
    overlap = df[in_train]
    print(f"--- {name} ---")
    print(f"Dòng trùng với train: {in_train.sum()} / {len(df)} ({in_train.mean():.1%})")
    print(f"Số câu khác nhau trong phần trùng: {overlap['free_text'].nunique()}")
    print("Phân bố nhãn của phần trùng:")
    print(overlap["label_id"].map(LABEL_NAMES).value_counts())
    print("Độ dài (số từ) của phần trùng:")
    print(overlap["free_text"].str.split().str.len().describe())
    print()

In [ ]:
# Với các câu test trùng train: nhãn ở test có khớp với (ít nhất một) nhãn trong train không?
# Nếu khớp, mô hình chỉ cần "nhớ" là đoán đúng -> điểm test bị thổi phồng
train_pairs = train_data[["free_text", "label_id"]].dropna().drop_duplicates()
test_overlap = test_data[test_data["free_text"].isin(train_texts)].reset_index()
merged = test_overlap.merge(train_pairs, on="free_text", suffixes=("", "_train"))
label_matches = (merged["label_id"] == merged["label_id_train"]).groupby(merged["index"]).any()
print(f"Câu test trùng train có nhãn khớp với train: {label_matches.sum()} / {len(label_matches)}")

# Câu trùng xuất hiện nhiều nhất trong test: câu dài, cụ thể hay ngắn, chung chung?
# (Mục đích: kiểm tra rò rỉ, không dùng để rút ra luật chuẩn hóa)
print(test_overlap["free_text"].value_counts().head(20))

In [ ]:
# Độ dài bình luận theo số từ (tách theo khoảng trắng). NaN tự bị bỏ qua trong describe()
for name, df in splits.items():
    n_words = df["free_text"].str.split().str.len()
    print(f"--- {name} ---")
    print(n_words.describe().round(1))
    # Phân vị cao: dùng để chọn max_length cho PhoBERT/ViSoBERT sau này
    print(n_words.quantile([0.9, 0.95, 0.99]))
    print()

In [ ]:
# Đọc 20 câu ngẫu nhiên cho MỖI nhãn, chỉ trên train (random_state cố định để tái lập)
# Không đọc mẫu dev/test: tránh để test ảnh hưởng tới quyết định (từ điển teencode, luật...)
for label_id, label_name in LABEL_NAMES.items():
    print(f"=== {label_name} ({label_id}) ===")
    sample = train_data[train_data["label_id"] == label_id].sample(20, random_state=42)
    print(sample["free_text"].to_string())
    print()

In [ ]:
for name, df in splits.items():
    print(f"Tập {name} có {(df['free_text'] == '#ERROR!').sum()} dòng là '#ERROR!'")